# 银行业务量预测：从 R 到 Python
## 未执行版 · 数据审计 → 基线复现 → 时间验证 → 有限优化 → 十月提交

本文件按你的最新要求**仅提供 Notebook，不运行实验**。所有代码单元均未执行；没有编造 Python 得分、模型改善或预测结果。按顺序运行后，图表、比较表、中文报告和提交文件会保存在 `bank_outputs/`。

### 使用方法 / README
1. 使用 Python 3.10 或更新版本、Jupyter Notebook / JupyterLab。
2. 安装依赖（在终端执行）：`pip install numpy pandas openpyxl matplotlib seaborn scikit-learn xgboost jupyter`。
3. 下方 `ZIP_PATH` 默认指向你的桌面压缩包，也可将压缩包放在 Notebook 同目录。代码仅提取所需数据和 R 脚本，不读取 `.RData`。
4. 从头顺序运行全部单元。实际软件版本及文件 SHA256 会记录到输出目录。依赖未锁定到未经验证的版本；首次成功运行后用生成的版本清单固定环境。

### 材料核对与范围
已阅读 `draft script.R` 和六张已有 EDA 图，并检查 Excel 表头：训练表并没有 `wkd_typ_cd`，该字段来自工作日历；测试表还有空的 `amount` 列，不能作为标签。
本次附带的 PDF 文件名为 `1-s2.0-S1359178915000348-main.pdf`，首页是其他主题的学术文章，并非任务说明中提到的比赛规则 PDF。因此评分口径依据你粘贴的任务说明和 R 脚本，尚未与官方 PDF 独立核对。

### R 审查要点
- `predict_b_recent()` 和 `b2` 在使用位置之后才定义；本文件将函数定义与实验分开，消除工作空间依赖。
- 原脚本对缺失业务量 `sum(..., na.rm=TRUE)`，可能把缺失混同为零。本文件遇到缺失或重复主键时停止并给出明细，不擅自补零或去重。
- 初始 Blend 对浮点预测融合后取整；后续 B 参数搜索对两个已取整分量再次融合取整；R 最终提交则使用浮点分量。本文件同时保留两种规则，量化差异。
- R 的随机种子、XGBoost 版本、默认 `base_score`、树构造方式及随机数实现可能使跨语言结果不同；没有 R 环境版本及逐折预测，不能保证逐位一致。
- R 最优成绩 11.033% 是历史参考，不是本文件已复现的结果。

### 验证纪律
7、8 月用于选择；选择结果写入 JSON 后才评估 9 月。9 月已被原 R 实验和前期分析使用，**只能称相对独立的时间检验，不是完全未见测试集**。不依据本次 9 月分数重新选模型。10 月没有真实标签，不能报告正式比赛成绩。


In [ ]:
from pathlib import Path
import json, zipfile, hashlib, platform, importlib.metadata, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor
from IPython.display import display

SEED = 2026
np.random.seed(SEED)
ZIP_PATH = Path.home() / 'Desktop' / '营销分析--HW02.zip'
if not ZIP_PATH.exists():
    ZIP_PATH = Path.cwd() / '营销分析--HW02.zip'
DATA = Path.cwd() / 'bank_data'
OUT = Path.cwd() / 'bank_outputs'
DATA.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
(OUT / 'figures').mkdir(exist_ok=True)
needed = {'train_v1-202001-202009.xlsx', 'test_v1.xlsx', 'wkd_v1.xlsx',
          'prediction_202010_optimized.xlsx', 'draft script.R'}
if ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        for member in z.infolist():
            name = Path(member.filename).name
            if name in needed:
                (DATA / name).write_bytes(z.read(member))
for name in ['train_v1-202001-202009.xlsx', 'test_v1.xlsx', 'wkd_v1.xlsx']:
    assert (DATA / name).exists(), f'缺少 {name}：检查 ZIP_PATH 或放入 {DATA}'
versions = {p: importlib.metadata.version(p) for p in
            ['numpy','pandas','openpyxl','matplotlib','seaborn','scikit-learn','xgboost']}
manifest = {'python': platform.python_version(), 'seed': SEED, 'versions': versions,
            'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in DATA.iterdir() if p.is_file()}}
(OUT / 'environment.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
(OUT / 'requirements_resolved.txt').write_text('\n'.join(f'{k}=={v}' for k,v in versions.items()),encoding='utf-8')
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei','SimHei','DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
display(versions)


## 1. 数据审计与日度聚合
原始粒度预期为 `(date, post_id, biz_type, periods)`。先检查必需字段、日期、金额、重复键与覆盖，再聚合。48 个时段覆盖仅作为审计表展示；若原始文件省略零量时段，不能在没有业务解释时认定为缺失。日度岗位键必须覆盖全部 274 天 × 2 岗位。


In [ ]:
def parse_dates(series):
    def one(v):
        if pd.isna(v):
            return pd.NaT
        if isinstance(v, (int,float,np.integer,np.floating)):
            return pd.Timestamp('1899-12-30') + pd.to_timedelta(float(v),unit='D')
        s = str(v).strip()
        if s.isdigit() and len(s)==8:
            return pd.to_datetime(s,format='%Y%m%d',errors='raise')
        if s.replace('.','',1).isdigit():
            return pd.Timestamp('1899-12-30') + pd.to_timedelta(float(s),unit='D')
        return pd.to_datetime(v,errors='raise')
    return series.map(one).pipe(pd.to_datetime).dt.normalize()

raw = pd.read_excel(DATA / 'train_v1-202001-202009.xlsx')
test_raw = pd.read_excel(DATA / 'test_v1.xlsx')
wkd = pd.read_excel(DATA / 'wkd_v1.xlsx')
assert {'date','post_id','biz_type','periods','amount'} <= set(raw)
assert {'date','post_id'} <= set(test_raw)
assert {'ORIG_DT','WKD_TYP_CD'} <= set(wkd)
raw['date'] = parse_dates(raw.date)
test_raw['date'] = parse_dates(test_raw.date)
for frame in [raw,test_raw]:
    frame['post_id'] = frame.post_id.astype('string').str.strip().str.upper()
raw['biz_type'] = raw.biz_type.astype('string').str.strip().str.upper()
raw['amount'] = pd.to_numeric(raw.amount,errors='raise')
raw['periods'] = pd.to_numeric(raw.periods,errors='raise')
cal = pd.DataFrame({'date':parse_dates(wkd.ORIG_DT),
                   'wkd_typ_cd':wkd.WKD_TYP_CD.astype('string').str.strip().str.upper()})
audit = pd.DataFrame({'missing':raw.isna().sum(),'dtype':raw.dtypes.astype(str)})
display(audit)
audit.to_csv(OUT/'data_audit.csv',encoding='utf-8-sig')
assert not raw[['date','post_id','biz_type','periods','amount']].isna().any().any()
assert not test_raw[['date','post_id']].isna().any().any()
assert not cal.isna().any().any()
assert np.isfinite(raw.amount).all() and raw.amount.ge(0).all()
assert raw.periods.isin(range(1,49)).all()
assert set(raw.post_id)=={'A','B'} and set(test_raw.post_id)=={'A','B'}
assert set(raw.loc[raw.post_id=='A','biz_type'])=={f'A{i}' for i in range(1,14)}
assert set(raw.loc[raw.post_id=='B','biz_type'])=={'B1'}
keys=['date','post_id','biz_type','periods']
dupes=raw[raw.duplicated(keys,keep=False)]
dupes.to_csv(OUT/'duplicate_keys.csv',index=False,encoding='utf-8-sig')
assert dupes.empty, '有重复原始主键，请检查 duplicate_keys.csv；未自动去重。'
assert not cal.date.duplicated().any()
assert cal.wkd_typ_cd.isin(['WN','WS','SN','SS','NH']).all()
assert not test_raw.duplicated(['date','post_id']).any()
coverage=raw.groupby(['date','post_id','biz_type']).agg(n_periods=('periods','nunique'),n_rows=('amount','size'))
coverage.to_csv(OUT/'period_coverage.csv',encoding='utf-8-sig')
display(coverage.groupby('post_id').agg(['min','max']))
if not coverage.n_periods.eq(48).all():
    warnings.warn('并非每个日期/业务都有48个时段，请核实是否采用稀疏零值存储；不补造交易。')
daily_base=raw.groupby(['date','post_id'],as_index=False).amount.sum()
expected=pd.MultiIndex.from_product([pd.date_range('2020-01-01','2020-09-30'),['A','B']],names=['date','post_id'])
assert daily_base.set_index(['date','post_id']).index.sort_values().equals(expected.sort_values())
test_keys=pd.MultiIndex.from_frame(test_raw[['date','post_id']])
expected_test=pd.MultiIndex.from_product([pd.date_range('2020-10-01','2020-10-31'),['A','B']],names=['date','post_id'])
assert len(test_raw)==62 and test_keys.sort_values().equals(expected_test.sort_values())
assert np.isclose(daily_base.amount.sum(),raw.amount.sum())
print('日度记录：',len(daily_base),'；测试记录：',len(test_raw))


## 2. 已知日历特征：与 R 对齐及新增特征
月内工作日序号、剩余工作日、节前节后休息天数可由预先公布的日历计算，不涉及未来业务量。训练与未来共用日历，但未来 `amount` 永远不传给模型。为避免整月多步预测泄露，本版不使用需要验证月真实值的滚动或滞后标签。


In [ ]:
cal=cal.sort_values('date').reset_index(drop=True)
assert cal.date.diff().dropna().eq(pd.Timedelta(days=1)).all(), '日历不连续'
assert cal.date.min()<=pd.Timestamp('2020-01-01') and cal.date.max()>=pd.Timestamp('2020-10-31')
cal['yr']=cal.date.dt.year
cal['month']=cal.date.dt.month
cal['dom']=cal.date.dt.day
cal['dow']=cal.date.dt.dayofweek+1
cal['day_of_year']=cal.date.dt.dayofyear
cal['time_index']=(cal.date-cal.date.min()).dt.days
cal['days_to_month_end']=cal.date.dt.days_in_month-cal.dom
cal['is_work']=cal.wkd_typ_cd.isin(['WN','WS']).astype(int)
for typ in ['WN','WS','SN','SS','NH']:
    cal['type_'+typ]=cal.wkd_typ_cd.eq(typ).astype(int)
g=cal.groupby(['yr','month'])['is_work']
cal['workday_number']=g.cumsum()
cal['workdays_left']=g.transform('sum')-cal.workday_number
cal['prev_break']=0
cal['next_break']=0
work=cal.is_work.to_numpy()
for i in range(len(cal)):
    if work[i]:
        j=i-1
        while j>=0 and work[j]==0:
            cal.loc[i,'prev_break']+=1
            j-=1
        j=i+1
        while j<len(cal) and work[j]==0:
            cal.loc[i,'next_break']+=1
            j+=1
cal['work_fraction']=cal.workday_number/(cal.workday_number+cal.workdays_left).clip(lower=1)
cal['quarter_end']=cal.month.isin([3,6,9,12]).astype(int)
cal['quarter_position']=cal.quarter_end*cal.work_fraction
cal['first_after_break']=cal.prev_break.ge(3).astype(int)
cal['last_before_break']=cal.next_break.ge(3).astype(int)
nh=cal.loc[cal.wkd_typ_cd.eq('NH'),'date']
cal['days_since_NH']=[min([30]+[(d-h).days for h in nh if h<=d]) for d in cal.date]
cal['days_until_NH']=[min([30]+[(h-d).days for h in nh if h>=d]) for d in cal.date]
R_FEATURES=['month','dom','dow','day_of_year','time_index','days_to_month_end',
            'workday_number','workdays_left','prev_break','next_break','is_work',
            'type_WN','type_WS','type_SN','type_SS','type_NH']
EXTRA_FEATURES=['work_fraction','quarter_end','quarter_position','first_after_break',
                'last_before_break','days_since_NH','days_until_NH']
daily=daily_base.merge(cal,on='date',how='left',validate='many_to_one').sort_values(['post_id','date']).reset_index(drop=True)
test=test_raw[['date','post_id']].assign(row_id=np.arange(len(test_raw))).merge(cal,on='date',how='left',validate='many_to_one').sort_values('row_id').reset_index(drop=True)
assert not daily[R_FEATURES].isna().any().any() and not test[R_FEATURES].isna().any().any()
daily.to_csv(OUT/'daily_aggregated.csv',index=False,encoding='utf-8-sig')
display(daily.groupby(['post_id','wkd_typ_cd']).amount.agg(['size','mean','min','max']))
display(daily.query("post_id=='B' and is_work==0").amount.describe())


## 3. 复现六张 EDA 图与业务假设
已提供的 R 图显示：2 月业务量显著降低，B 在 8 月走低、9 月回升；A12 与 A8 贡献约 60.9% 与 20.3%；特殊日类型的样本很少（例如调休工作日仅 5 天）。这些是历史描述，不是已验证因果关系。

以下默认只展示截至 8 月的探索，9 月在冻结方案后诊断。可将 `EDA_END` 设为 `2020-09-30` 复现 R 的完整范围，但不要再把 9 月称为未见数据。月末增长与季度效应仅是候选假设；不强加 early/middle/late 乘数。日内图不作为十月已知业务特征。


In [ ]:
EDA_END=pd.Timestamp('2020-08-31')
eda=daily[daily.date<=EDA_END].copy()
eraw=raw[raw.date<=EDA_END].copy()
def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT/'figures'/name,dpi=160,bbox_inches='tight')
    plt.show()
fig,axes=plt.subplots(2,1,figsize=(12,7))
for ax,(post,h) in zip(axes,eda.groupby('post_id')):
    ax.plot(h.date,h.amount,alpha=.4,label='每日业务量')
    ax.plot(h.date,h.amount.rolling(7).mean(),label='7日均量')
    ax.set_title(post); ax.legend()
savefig('01_daily_trend.png')
for column,name in [('wkd_typ_cd','02_calendar_type.png'),('dow','03_weekday_effect.png')]:
    tab=eda.groupby(['post_id',column]).amount.agg(['mean','size']).reset_index()
    fig,axes=plt.subplots(2,1,figsize=(10,7))
    for ax,post in zip(axes,['A','B']):
        d=tab[tab.post_id==post]
        ax.bar(d[column].astype(str),d['mean'])
        for i,row in enumerate(d.itertuples()):
            ax.text(i,row.mean,f'n={row.size}',ha='center',va='bottom')
        ax.set_title(post)
    savefig(name)
slots=eraw.groupby(['date','post_id','periods'],as_index=False).amount.sum()
intraday=slots.groupby(['post_id','periods'],as_index=False).amount.mean()
intraday['hour']=(intraday.periods-.5)/2
fig,axes=plt.subplots(2,1,figsize=(10,7))
for ax,post in zip(axes,['A','B']):
    h=intraday[intraday.post_id==post]
    ax.plot(h.hour,h.amount); ax.set_title(post)
savefig('04_intraday_pattern.png')
share=eraw.query("post_id=='A'").groupby('biz_type').amount.sum().sort_values()
share=100*share/share.sum()
share.plot.barh(figsize=(9,5),title='A业务类型贡献（%）')
savefig('05_business_composition.png')
monthly=eda.groupby(['post_id','month']).amount.mean().unstack('post_id')
monthly.plot(subplots=True,figsize=(10,6),marker='o',title='月度平均每日业务量')
savefig('06_monthly_trend.png')
b=eda.query("post_id=='B' and is_work==1 and month>=4").copy()
b['block']=np.ceil(b.workday_number/5).astype(int)
blocks=b.groupby(['month','block'],as_index=False).amount.mean()
sns.lineplot(data=blocks,x='block',y='amount',hue='month',palette='tab10',marker='o')
savefig('07_within_month.png')
growth=[]
for month,h in b.groupby('month'):
    first,last=h.head(5).amount.mean(),h.tail(5).amount.mean()
    growth.append({'month':month,'First5':first,'Last5':last,'growth_pct':100*(last/first-1) if first else np.nan})
display(pd.DataFrame(growth))
print('月内增长不等同季度末因果效应；现有季度样本不足。')


## 4. 统一评分、后处理及 R 基准
官方口径（依任务说明）：`100 × mean(abs(y-p)/(y+1))`。A 全日期；B 仅 WN/WS；综合为两个岗位 MAPE 等权平均。跨月先合并每个岗位的逐日误差再计算，不能直接平均月度 MAPE。

`np.rint` 与 R `round` 均采用遇半取偶；浮点边界仍可能有微小差异。B 非工作日置零是沿用 R 的业务假设，评分本身仅表示不计分，不自动证明其真实业务量为零。


In [ ]:
def postprocess(pred,future):
    p=np.maximum(0,np.rint(np.asarray(pred,dtype=float)))
    p[(future.post_id.to_numpy()=='B') & (future.is_work.to_numpy()==0)]=0
    assert np.isfinite(p).all()
    return p.astype(np.int64)

def metrics(frame):
    z=frame[(frame.post_id=='A') | frame.is_work.eq(1)]
    assert len(z)>0 and z.amount.notna().all() and z.pred.notna().all()
    e=z.pred-z.amount
    return {'n':len(z),'MAPE':float((e.abs()/(z.amount+1)).mean()*100),
            'Bias':float(e.mean()),'MAE':float(e.abs().mean()),
            'Actual_Mean':float(z.amount.mean()),'Predict_Mean':float(z.pred.mean())}

def score_table(frame,groups):
    records=[]
    for key,h in frame.groupby(groups,sort=False):
        if not isinstance(key,tuple): key=(key,)
        records.append(dict(zip(groups,key)) | metrics(h))
    return pd.DataFrame(records)

def competition_score(frame):
    scores=score_table(frame,['post_id']).set_index('post_id')
    assert {'A','B'}<=set(scores.index)
    return float(scores.loc[['A','B'],'MAPE'].mean())

def recent(history,future,k=8,b_mode=False):
    h=history.sort_values('date')
    if h.post_id.iloc[0]=='B': h=h[h.is_work==1]
    result=[]
    for row in future.itertuples():
        if b_mode and row.is_work==0:
            result.append(0.); continue
        pool=h[h.wkd_typ_cd==row.wkd_typ_cd]
        if b_mode:
            pool=pool[pool.dow==row.dow]
            if len(pool)<2: pool=h[h.dow==row.dow]
        elif row.wkd_typ_cd in ['WN','SN']:
            same=pool[pool.dow==row.dow]
            if len(same)>=3: pool=same
        if pool.empty: pool=h
        pool=pool.tail(k)
        result.append(np.average(pool.amount,weights=np.arange(1,len(pool)+1)))
    return np.array(result)

def level_baseline(history,future,window=20):
    h=history[history.is_work==1].sort_values('date')
    r=h.tail(window); season=h.tail(80)
    level=np.average(r.amount,weights=np.arange(1,len(r)+1))
    overall=season.amount.mean()
    factor=(season.groupby('dow').amount.mean()/overall).clip(.7,1.3) if overall>0 else pd.Series(dtype=float)
    return np.where(future.is_work==1,level*future.dow.map(factor).fillna(1).to_numpy(),0)

def tree_predict(history,future,kind='xgb_r',enhanced=False,tau=90,weighted=True):
    h=history.copy()
    if h.post_id.iloc[0]=='B': h=h[h.is_work==1]
    features=R_FEATURES+(EXTRA_FEATURES if enhanced else [])
    X=h[features].to_numpy(dtype=float); F=future[features].to_numpy(dtype=float)
    y=np.log1p(h.amount.to_numpy())
    age=(h.date.max()-h.date).dt.days.to_numpy()
    weights=.3+.7*np.exp(-age/tau) if weighted else np.ones(len(h))
    if kind=='hgb':
        model=HistGradientBoostingRegressor(max_iter=180,max_leaf_nodes=7,
            learning_rate=.04,l2_regularization=3,min_samples_leaf=15,
            early_stopping=False,random_state=SEED)
        model.fit(X,y,sample_weight=weights)
        pred=model.predict(F)
    else:
        params=dict(objective='reg:squarederror',max_depth=3,eta=.04,
            min_child_weight=4,subsample=.9,colsample_bytree=.9,
            reg_lambda=3,nthread=2,seed=SEED,tree_method='hist',base_score=.5)
        model=xgb.train(params,xgb.DMatrix(X,label=y,weight=weights),num_boost_round=220)
        pred=model.predict(xgb.DMatrix(F))
    return np.maximum(0,np.expm1(pred))


## 5. 有限候选集与消融设计
只尝试适合小样本的方案：近期基线、R 风格 XGBoost、近期水平、HistGradientBoosting。固定 `hist` 和 `base_score=0.5` 使 Python 配置明确，但旧 R 默认值未知，这属于复现差异来源，不能将本实现称为字节级复现。

- A：Recent-8，XGB，50/50 融合，以及少量树模型变体。
- B：Recent-k（2/4/6/8/12）、近期 20 工作日水平与星期效应、受约束融合权重。
- 配对消融：新增日历特征、去掉近期加权、权重时间尺度 90→45；每次只改一个因素。
- 不重试已失败的经验乘数；不盲目增加算法。A 分业务预测暂未加入：对相同线性基线和完整业务覆盖，分项再汇总常与总量预测等价；不同模型虽可能获益，也增加估计方差。


In [ ]:
def candidate_predictions(history,future):
    assert 'amount' not in future.columns, '未来特征不得携带真实标签'
    assert history.date.max()<future.date.min()
    post=history.post_id.iloc[0]
    pr=recent(history,future)
    px=tree_predict(history,future)
    out={'recent8':pr,'xgb_r':px,'blend50_raw':.5*px+.5*pr,
         'xgb_calendar':tree_predict(history,future,enhanced=True),
         'xgb_unweighted':tree_predict(history,future,weighted=False),
         'xgb_tau45':tree_predict(history,future,tau=45),
         'hgb_calendar':tree_predict(history,future,kind='hgb',enhanced=True)}
    if post=='B':
        components={f'b_recent{k}':recent(history,future,k=k,b_mode=True) for k in [2,4,6,8,12]}
        components['level20']=level_baseline(history,future)
        out.update(components)
        for name,base in components.items():
            for weight in [.25,.5,.75]:
                out[f'{name}_xgb{weight:g}']=(1-weight)*base+weight*px
        # R 后期搜索使用先取整分量；单独保留用于历史对照，不参与新方案选择
        out['R_best_rounded']=.5*postprocess(px,future)+.5*postprocess(components['b_recent12'],future)
        out['R_best_raw']=.5*px+.5*components['b_recent12']
    return {name:postprocess(pred,future) for name,pred in out.items()}

def backtest(starts):
    results=[]
    for start in pd.to_datetime(starts):
        end=start+pd.offsets.MonthBegin(1)
        for post in ['A','B']:
            h=daily[(daily.post_id==post)&(daily.date<start)].copy()
            v=daily[(daily.post_id==post)&(daily.date>=start)&(daily.date<end)].copy()
            assert not h.empty and not v.empty
            # 整个月统一预测：模型函数只收到日期特征，不收到验证标签
            predictions=candidate_predictions(h,v.drop(columns='amount'))
            for name,p in predictions.items():
                out=v[['date','post_id','amount','is_work','workday_number','wkd_typ_cd']].copy()
                out['fold']=start.strftime('%Y-%m'); out['model']=name; out['pred']=p
                results.append(out)
    return pd.concat(results,ignore_index=True)

cv78=backtest(['2020-07-01','2020-08-01'])
scores78=score_table(cv78,['post_id','model']).sort_values(['post_id','MAPE'])
monthly78=score_table(cv78,['fold','post_id','model'])
scores78.to_csv(OUT/'selection_78.csv',index=False,encoding='utf-8-sig')
monthly78.to_csv(OUT/'monthly_78.csv',index=False,encoding='utf-8-sig')
display(scores78)
# 仅用7-8月，固定0.10个百分点最小改善门槛；门槛未从9月调整。
fallback={'A':'recent8','B':'R_best_raw'}
selected={}
for post in ['A','B']:
    tab=scores78[(scores78.post_id==post)&(~scores78.model.eq('R_best_rounded'))].sort_values(['MAPE','model'])
    best=tab.iloc[0]
    ref=float(tab.loc[tab.model==fallback[post],'MAPE'].iloc[0])
    selected[post]=str(best.model) if best.MAPE<ref-.10 else fallback[post]
selection_record={'selected':selected,'selection_months':['2020-07','2020-08'],
                  'minimum_improvement_pp':.10,'seed':SEED,'fallback':fallback}
(OUT/'frozen_selection.json').write_text(json.dumps(selection_record,ensure_ascii=False,indent=2),encoding='utf-8')
display(selected)


## 6. 冻结后评估九月：回测、复现差异与漂移诊断
下方对全部预先声明候选保存 9 月分数用于消融解释，但最终选择仍来自上面的 7–8 月 JSON。若之后据此调参，必须将 9 月重新标记为开发集，并寻找新的未见时期，不能继续声称独立检验。


In [ ]:
selected=json.loads((OUT/'frozen_selection.json').read_text(encoding='utf-8'))['selected']
cv9=backtest(['2020-09-01'])
cv=pd.concat([cv78,cv9],ignore_index=True)
cv.to_csv(OUT/'all_oof_predictions.csv',index=False,encoding='utf-8-sig')
all_scores=score_table(cv,['post_id','model'])
month_scores=score_table(cv,['fold','post_id','model'])
month_scores.to_csv(OUT/'monthly_all_models.csv',index=False,encoding='utf-8-sig')
chosen=cv[cv.model.eq(cv.post_id.map(selected))].copy()
assert not chosen.duplicated(['date','post_id']).any()
chosen_month=score_table(chosen,['fold','post_id'])
combined=chosen_month.pivot(index='fold',columns='post_id',values='MAPE')
combined['综合MAPE']=(combined.A+combined.B)/2
display(chosen_month); display(combined)
overall=competition_score(chosen)
september=competition_score(chosen[chosen.fold=='2020-09'])
print(f'7-9月总体回测综合MAPE：{overall:.3f}%')
print(f'9月相对独立时间检验综合MAPE：{september:.3f}%')
print('9月结果不触发重新选模。')
reference=pd.DataFrame([
    ['A','xgb_r',7.936],['B','xgb_r',18.180],
    ['A','recent8',6.606],['B','recent8',16.816],
    ['A','blend50_raw',6.955],['B','blend50_raw',16.323],
    ['B','R_best_rounded',15.459]],columns=['post_id','model','R_reference_MAPE'])
reproduction=reference.merge(all_scores[['post_id','model','MAPE']],on=['post_id','model'],validate='one_to_one')
reproduction['delta_pp']=reproduction.MAPE-reproduction.R_reference_MAPE
reproduction.to_csv(OUT/'R_reproduction.csv',index=False,encoding='utf-8-sig')
display(reproduction)
print('差异排查顺序：数据覆盖/缺失 → 工作日过滤 → 特征顺序 → 分量取整 → XGBoost版本与默认值/随机性。')
print('仅当取得R逐折预测与sessionInfo后，才能进一步定位XGB数值差异。')
ablation=month_scores[month_scores.model.isin(['xgb_r','xgb_calendar','xgb_unweighted','xgb_tau45','hgb_calendar','recent8','level20','R_best_raw','R_best_rounded'])]
ablation.to_csv(OUT/'ablation.csv',index=False,encoding='utf-8-sig')
display(ablation.pivot_table(index=['post_id','model'],columns='fold',values='MAPE'))
# 相同月份相同岗位配对差异：负值表示改动降低MAPE。
paired=ablation.merge(month_scores[month_scores.model.eq('xgb_r')][['fold','post_id','MAPE']].rename(columns={'MAPE':'XGB_base_MAPE'}),on=['fold','post_id'])
paired['delta_vs_xgb_pp']=paired.MAPE-paired.XGB_base_MAPE
paired.to_csv(OUT/'ablation_paired.csv',index=False,encoding='utf-8-sig')
bdiag=chosen.query("post_id=='B' and is_work==1").copy()
bdiag['block']=np.ceil(bdiag.workday_number/5).astype(int)
display(score_table(bdiag,['fold','block']))
fig,axes=plt.subplots(3,1,figsize=(12,9))
for ax,(fold,h) in zip(axes,bdiag.groupby('fold')):
    ax.plot(h.date,h.amount,label='实际值',marker='o')
    ax.plot(h.date,h.pred,label='预测值',marker='.')
    ax.set_title('B '+fold); ax.legend()
savefig('08_B_monthly_diagnosis.png')
chosen_month.to_csv(OUT/'selected_monthly_metrics.csv',index=False,encoding='utf-8-sig')
combined.to_csv(OUT/'selected_monthly_combined.csv',encoding='utf-8-sig')


## 7. 使用全部训练数据预测十月，并与现有 R 提交对照
最终方案由 7–8 月选择并冻结。全部训练标签截至 9 月 30 日；整月预测一次完成，不读取十月真实业务量。输出使用遇半取偶的非负整数，保留测试表原始 62 行顺序。历史 R 提交仅用于逐行差异诊断，不能用于挑选十月预测或计算准确率。

稀少假日样本可能导致 A 在连续日期预测相同。下面导出假日日历、同类历史样本数量和预测范围，重复数值本身不触发人工修改。默认方案不保证优于历史 11.033%；若运行后改善仅来自单月，应在报告中保留这一限制。


In [ ]:
prediction=np.full(len(test),np.nan)
r_style_prediction=np.full(len(test),np.nan)
for post in ['A','B']:
    h=daily[daily.post_id==post].copy()
    mask=test.post_id.eq(post)
    f=test.loc[mask].copy()
    bank=candidate_predictions(h,f)
    prediction[mask.to_numpy()]=bank[selected[post]]
    r_style_prediction[mask.to_numpy()]=bank['recent8' if post=='A' else 'R_best_raw']
assert np.isfinite(prediction).all() and (prediction>=0).all()
result=test[['date','post_id']].copy()
result['Predict_amount']=prediction.astype(np.int64)
assert list(result)==['date','post_id','Predict_amount'] and len(result)==62
pd.testing.assert_frame_equal(result[['date','post_id']].reset_index(drop=True),test_raw[['date','post_id']].reset_index(drop=True))
assert result.loc[(test.post_id=='B')&test.is_work.eq(0),'Predict_amount'].eq(0).all()
result.to_excel(OUT/'prediction_202010_python.xlsx',index=False)
r_style=result.copy(); r_style['Predict_amount']=r_style_prediction.astype(np.int64)
r_style.to_excel(OUT/'prediction_202010_R_style_python.xlsx',index=False)
review=test[['date','post_id','wkd_typ_cd','prev_break','next_break','workday_number']].copy()
review['Predict_amount']=prediction.astype(np.int64)
counts=daily.groupby(['post_id','wkd_typ_cd']).size().rename('historical_type_n').reset_index()
review=review.merge(counts,on=['post_id','wkd_typ_cd'],how='left')
review.to_csv(OUT/'October_calendar_review.csv',index=False,encoding='utf-8-sig')
display(review[review.date.le('2020-10-12')])
r_path=DATA/'prediction_202010_optimized.xlsx'
if r_path.exists():
    old=pd.read_excel(r_path)
    old['date']=parse_dates(old.date)
    old['post_id']=old.post_id.astype(str).str.strip()
    old=old.rename(columns={'Predict_amount':'R_Predict_amount'})
    compare=result.merge(old[['date','post_id','R_Predict_amount']],on=['date','post_id'],how='left',validate='one_to_one')
    assert compare.R_Predict_amount.notna().all()
    compare['difference']=compare.Predict_amount-compare.R_Predict_amount
    compare.to_csv(OUT/'October_R_comparison.csv',index=False,encoding='utf-8-sig')
    display(compare.head(16))
    print('十月差异仅反映模型输出不同，不代表哪一个更准确。')
print('提交文件：',(OUT/'prediction_202010_python.xlsx').resolve())


## 8. 自动生成中文结果摘要与课堂展示提纲
报告只引用本次实际运行产生的结果。结论要结合分月误差及消融，不能将总体回测降低等同于十月一定更好。当前 Notebook 未运行，因此以下报告也尚未生成。


In [ ]:
reference_frame=cv[cv.model.eq(cv.post_id.map({'A':'recent8','B':'R_best_rounded'}))]
reference_python=competition_score(reference_frame)
delta=overall-11.033
direction='低于' if delta<0 else '不低于'
report=f"""# 银行业务量预测：Python 实验报告

## 数据与方法
原始记录 {len(raw):,} 行，聚合后 {len(daily)} 条日度岗位记录，测试表 {len(test)} 行。
评分为 100×mean(abs(actual-pred)/(actual+1))，B仅评分工作日；综合对A/B等权平均。
函数不接收验证期标签，整月统一预测，训练使用当月之前的全部历史。
最终选择：A={selected['A']}；B={selected['B']}。仅使用7-8月选择，9月不重新调参。

## 实际运行结果
7-9月总体综合MAPE：{overall:.3f}%。
9月相对独立时间检验综合MAPE：{september:.3f}%。
Python实现的R固定方案回测综合MAPE：{reference_python:.3f}%。
本次总体回测{direction}历史R参考11.033%，差值（Python−R）为{delta:+.3f}个百分点。
历史成绩与本次结果并非完全相同的选模流程，不能仅凭这一差值断言泛化改善。

## 分月诊断
{chosen_month.to_string(index=False)}

## 复现检查
{reproduction.to_string(index=False)}

## 消融与失败实验
详见 ablation.csv 和 ablation_paired.csv；应同时检查7、8、9月的改进方向与幅度。
若改动只改善开发月份而在9月恶化，应明确记录其不稳定性，不事后更换已冻结方案。
R曾尝试月内阶段乘数，历史B MAPE由15.459%升到15.626%；本次没有重试或默认启用该规则。

## 业务解释与局限
工作日/休息日、星期与月份水平变化决定预测难度。月内增长只作为待检验假设。
2月异常与8、9月漂移并不能仅凭图形确认原因；季度末可用月份少，无法识别稳健因果效应。
9月已经被前期R分析观察，故不是完全盲测。调休与长假样本稀少，十月外推不确定性大。
XGBoost跨语言版本、默认参数、随机性与取整顺序可导致复现差异。
原始比赛规则PDF未获得；评分按提供说明实现。没有十月真实标签，不报告十月准确率。

## 交付索引
prediction_202010_python.xlsx：正式格式的十月预测。
prediction_202010_R_style_python.xlsx：Python实现R固定策略的参考预测。
environment.json、requirements_resolved.txt：输入哈希与运行版本。
all_oof_predictions.csv：所有候选的逐日样本外预测。
frozen_selection.json：7-8月冻结选模记录。
figures/：EDA与误差诊断图。
"""
(OUT/'中文技术报告.md').write_text(report,encoding='utf-8')
display(result.head())
print(report)


### 15 分钟课堂汇报提纲（2026 年 11 月 18 日）
| 时间 | 内容 | 建议图表与讲述重点 |
|---|---|---|
| 0–2 分钟 | 任务与指标 | 每日A/B总量、整月预测；分母+1与岗位等权，B只计工作日 |
| 2–4 分钟 | 数据理解 | 日度趋势、日期类型、A12/A8贡献；区分低业务量与缺失 |
| 4–6 分钟 | 为什么从基线开始 | 同类日加权均值、R工作流修复、样本量与复杂度 |
| 6–9 分钟 | 如何验证优化 | 扩展窗口示意、7–8月选择与9月检验、特征/权重配对消融 |
| 9–12 分钟 | 哪些误差仍未解决 | B分月实际/预测、8月高估与9月低估是否在本次仍存在；按真实结果讲述 |
| 12–14 分钟 | 十月方案为什么可信 | 冻结模型、无未来标签、原行序、国庆与调休检查 |
| 14–15 分钟 | 局限与后续 | 稀少假日样本、月际漂移、9月非完全盲测、无十月真实成绩 |

**运行前检查清单**：数据路径正确、依赖已装、从空内核顺序执行。  
**运行后检查清单**：读取审计表、比较 R 复现差值、查看分月 Bias/MAE、确认选模 JSON 在九月评估前冻结、核验 Excel 的 62 行三列。  
**尚未验证的事项**：本文件只做了静态语法和 Notebook 结构检查；依赖兼容性、实际数值复现与运行时行为需要执行后确认。
